# 🛢️ OilyGiant: Qual a melhor região para novos poços de petróleo
Nesse projeto, estarei atuando como uma funcionária de uma empresa de mineração , cujo objetivo é encontrar os melhores lugares para o desenvolvimento de novos poços de petróleo.  

Para realizar essa tarefa vou executar os seguintes etapas:
1. Carregamento e preparação dos dados
2. Treinamento e validação do modelo por região
3. Preparação para o cálculo de lucro
4. Cálculo de lucro com os 200 melhores poços
5. Riscos e lucro com bootstrapping
6. Elaborar conclusão geral

Há algumas condições que preciso considerar para encontrar os melhores lugares:
- Apenas regressão linear deve ser usada para o treinamento do modelo.
- Ao explorar a região, um estudo de 500 pontos é realizado e os melhores 200 pontos são selecionados para calcular o lucro.
- O orçamento para o desenvolvimento de 200 poços de petróleo é 100 milhões de dólares.
- Um barril de petróleo bruto traz 4.5 dólares de receita. A receita de uma unidade de produto é 4.500 dólares (o volume de reservas está em milhares de barris).
- Depois de ter avaliado os riscos, mantenha apenas as regiões com o risco de perdas inferior a 2.5%. Entre aquelas que se enquadram no critério, você precisa selecionar a região com o lucro médio mais alto.



## 1. Carregamento e preparação dos dados
Primeiramente, vou carregar os dados fornecidos e examinar os conjuntos de dados para verificar:
- Dados ausentes
- Dados duplicados
- Formatação
Serão carregados três conjuntos de dados representando três regiões e agrupadas em um dicionários chamado `data` (usando uma função `load_data`). Isso permite que que os dados quando exibidos estejam ligados a cada região  e evita duplicação de códigos.

In [20]:
# importar as bibliotecas necessárias
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 12345

In [21]:
# Função para carregar os dados do CSV em arquivos separados por região
def load_data(filename):
    """Lê o CSV tentando o caminho da plataforma e depois o caminho local."""
    try:
        return pd.read_csv(f'dataset/{filename}')
    except FileNotFoundError:
        return pd.read_csv(filename)

files = {'Região 0': 'geo_data_0.csv',
         'Região 1': 'geo_data_1.csv',
         'Região 2': 'geo_data_2.csv'}

data = {region: load_data(file) for region, file in files.items()}

### 1.1  Inspeção inicial dos dados

In [22]:
# função para examinar os dados e verificar se há valores ausentes
for region, df in data.items():
    print(f"Região: {region}")
    print(f"Dimensões do DataFrame: {df.shape}")
    df.info()
    display(df.head())
    print('Valores nulos encontrados:',df.isnull().sum().sum())
    print('Linhas duplicadas encontradas:', df.duplicated().sum())
    print("IDs duplicados encontrados:", df['id'].duplicated().sum())
    print(df.describe().round(2))
    print("\n")

Região: Região 0
Dimensões do DataFrame: (100000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB


,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647


Valores nulos encontrados: 0
Linhas duplicadas encontradas: 0
IDs duplicados encontrados: 10
              f0         f1         f2    product
count  100000.00  100000.00  100000.00  100000.00
mean        0.50       0.25       2.50      92.50
std         0.87       0.50       3.25      44.29
min        -1.41      -0.85     -12.09       0.00
25%        -0.07      -0.20       0.29      56.50
50%         0.50       0.25       2.52      91.85
75%         1.07       0.70       4.72     128.56
max         2.36       1.34      16.00     185.36


Região: Região 1
Dimensões do DataFrame: (100000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memo

,id,f0,f1,f2,product
0,kBEdx,-15.001348,-8.276000,-0.005876,3.179103
1,62mP7,14.272088,-3.475083,0.999183,26.953261
2,vyE1P,6.263187,-5.948386,5.001160,134.766305
3,KcrkZ,-13.081196,-11.506057,4.999415,137.945408
4,AHL4O,12.702195,-8.147433,5.004363,134.766305


Valores nulos encontrados: 0
Linhas duplicadas encontradas: 0
IDs duplicados encontrados: 4
              f0         f1         f2    product
count  100000.00  100000.00  100000.00  100000.00
mean        1.14      -4.80       2.49      68.82
std         8.97       5.12       1.70      45.94
min       -31.61     -26.36      -0.02       0.00
25%        -6.30      -8.27       1.00      26.95
50%         1.15      -4.81       2.01      57.09
75%         8.62      -1.33       4.00     107.81
max        29.42      18.73       5.02     137.95


Região: Região 2
Dimensões do DataFrame: (100000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memor

,id,f0,f1,f2,product
0,fwXo0,-1.146987,0.963328,-0.828965,27.758673
1,WJtFt,0.262778,0.269839,-2.530187,56.069697
2,ovLUW,0.194587,0.289035,-5.586433,62.871910
3,q6cA6,2.236060,-0.553760,0.930038,114.572842
4,WPMUX,-0.515993,1.716266,5.899011,149.600746


Valores nulos encontrados: 0
Linhas duplicadas encontradas: 0
IDs duplicados encontrados: 4
              f0         f1         f2    product
count  100000.00  100000.00  100000.00  100000.00
mean        0.00      -0.00       2.50      95.00
std         1.73       1.73       3.47      44.75
min        -8.76      -7.08     -11.97       0.00
25%        -1.16      -1.17       0.13      59.45
50%         0.01      -0.01       2.48      94.93
75%         1.16       1.16       4.86     130.60
max         7.24       7.84      16.74     190.03




**COMENTÁRIOS:** Examinamos os conjuntos de dados das três regiões não encontramos: dados ausentes ou formatos inconsistentes, porém detectamos ID (identificador unívoco de poço de petróleo) duplicados. Como não detectamos linhas que tenham dados totalmente duplicados, só os ID, vamos avaliar com maior detalhamento esses casos

#### 1.1.4 Inspeção de ID duplicados

In [23]:
# iterando sobre cada região para encontrar IDs duplicados
for region, df in data.items():
    dup_ids=df[df['id'].duplicated(keep=False)].sort_values(by='id')
    print(f'{region}: {dup_ids["id"].nunique()} IDS repetidos encontrados, {len(dup_ids)} linhas duplicadas encontradas.')
    display(dup_ids)

Região 0: 10 IDS repetidos encontrados, 20 linhas duplicadas encontradas.


,id,f0,f1,f2,product
66136,74z30,1.084962,-0.312358,6.990771,127.643327
64022,74z30,0.741456,0.459229,5.153109,140.771492
51970,A5aEY,-0.180335,0.935548,-2.094773,33.020205
3389,A5aEY,-0.039949,0.156872,0.209861,89.249364
69163,AGS9W,-0.933795,0.116194,-3.655896,19.230453
42529,AGS9W,1.454747,-0.479651,0.683380,126.370504
931,HZww2,0.755284,0.368511,1.863211,30.681774
7530,HZww2,1.061194,-0.373969,10.430210,158.828695
63593,QcMuo,0.635635,-0.473422,0.862670,64.578675
1949,QcMuo,0.506563,-0.323775,-2.215583,75.496502


Região 1: 4 IDS repetidos encontrados, 8 linhas duplicadas encontradas.


,id,f0,f1,f2,product
5849,5ltQ6,-3.435401,-12.296043,1.999796,57.085625
84461,5ltQ6,18.213839,2.191999,3.993869,107.813044
1305,LHZR0,11.170835,-1.945066,3.002872,80.859783
41906,LHZR0,-8.989672,-4.286607,2.009139,57.085625
2721,bfPNe,-9.494442,-5.463692,4.006042,110.992147
82178,bfPNe,-6.202799,-4.820045,2.995107,84.038886
47591,wt4Uk,-9.091098,-8.109279,-0.002314,3.179103
82873,wt4Uk,10.259972,-9.376355,4.994297,134.766305


Região 2: 4 IDS repetidos encontrados, 8 linhas duplicadas encontradas.


,id,f0,f1,f2,product
45404,KUPhW,0.231846,-1.698941,4.990775,11.716299
55967,KUPhW,1.211150,3.176408,5.543540,132.831802
11449,VF7Jo,2.122656,-0.858275,5.746001,181.716817
49564,VF7Jo,-0.883115,0.560537,0.723601,136.233420
44378,Vcm5J,-1.229484,-2.439204,1.222909,137.968290
95090,Vcm5J,2.587702,1.986875,2.482245,92.327572
28039,xCHr8,1.633027,0.368135,-2.378367,6.120525
43233,xCHr8,-0.847066,2.101796,5.597130,184.388641


**DECISÃO:** Após, avaliarmos cada linha que teve o ID duplicados constatamos que registros com mesmo ´id´ não são consequência de um erro de duplicação de dados ou de erros de digitação. São linhas de poços diferentes que receberam o mesmo código: cada par tem valores diferentes de características e de `product`. Ao verificar os valores os registro mostram coerência de valores. Como o valor `id` é apenas identificador a retirada de linhas seria perder informações. Desta forma as linhas foram mantidas.

### 1.2 Correlação das características com a variável objetivo
O objetivo para fazer uma correlação das características com a variável objetivo antes de treinar os modelos para verificar as relações ente `f0`, `f1`,`f2` com o volume das reservas (`product). Como usaremos uma regressão linear para modelo, ele captura relações lineares entre as variáveis.A correlação de Pearson uma medida adequada para antecipar o desempenho esperado do modelo em cada região: se a correlação foi alta, menor a tendencia de erros na predição. Essa etapa fornece uma bases para interpretar, mais adiante, as diferenças de REQM entre regiões e, as diferenças de riscos e lucro obtidas no bootstrapping

In [24]:
# Iteração para correlação entre as variáveis
for region, df in data.items():
    print(f"Região: {region}")
    display(df.drop(columns=['id']).corr()['product'].round(3).to_frame("Correlação com a variável alvo"))

Região: Região 0


,Correlação com a variável alvo
f0,0.144
f1,-0.192
f2,0.484
product,1.000


Região: Região 1


,Correlação com a variável alvo
f0,-0.030
f1,-0.010
f2,0.999
product,1.000


Região: Região 2


,Correlação com a variável alvo
f0,-0.002
f1,-0.001
f2,0.446
product,1.000


### Observações sobre etapa de preparação dos dados
Nas três regiões não foram encontrados valores ausentes, ne formatos de dados incoerentes. Identificamos `id` repetidos em todos os conjuntos de dados, mas verificamos que os registros se tratavam e poços distintos, com valores de características e de `product` diferentes e coerentes entre si. Por isso, as linhas foram mantidas.

A analise de correlação mostrou que, nas regiões 0 e 2, a característica `f2` tem correlação moderada com `product`(0,48 e 0,45 respectivamente), enquanto para as outras características a correlação é fraca ou nula. Na região1 , `f2` tem uma correlação praticamente perfeita com `product`(~ 0.999), indicando uma relação quase linear. Como o modelo exigido é uma regressão linear, espera-se que o erro de predição (REQM) seja muito menor na Região 1 do que nas demais.

## 2. Treinamento e teste do modelo por região

### 2.1 Região 0: divisão dos dados em treino e validação(75:25)

In [25]:
df_0 = data['Região 0']

features = df_0.drop(columns=['id', 'product'])
target = df_0['product']

features_train_0, features_valid_0, target_train_0, target_valid_0 = train_test_split(features, target, test_size=0.25, random_state=RANDOM_STATE)

# Padronização dos dados
scaler = StandardScaler().fit(features_train_0)
features_train_0_scaled = scaler.transform(features_train_0)
features_valid_0_scaled = scaler.transform(features_valid_0)    

print("Dimensões do conjunto de treino:", features_train_0_scaled.shape)
print("Dimensões do conjunto de validação:", features_valid_0_scaled.shape)


Dimensões do conjunto de treino: (75000, 3)
Dimensões do conjunto de validação: (25000, 3)


### 2.2 Região 0: treino do modelo

In [26]:
model_0 = LinearRegression().fit(features_train_0_scaled, target_train_0)
predictions_0 = model_0.predict(features_valid_0_scaled)

#Salvar previsões do modelo
# Guardados em arrays alinhados por posição (usados nas etapas 4 e 5)
target_valid_0 = target_valid_0.to_numpy()

# volume médio previsto e REQM
rmse_0 = mean_squared_error(target_valid_0, predictions_0) ** 0.5

print(f'Volume médio previsto: {predictions_0.mean():.2f} mil barris')
print(f'Volume médio real:     {target_valid_0.mean():.2f} mil barris')
print(f'REQM:                  {rmse_0:.2f} mil barris')

#Modelo constante: prevê a média do treino para 
print("Verificação de  sanidade:")
# Modelo constante: prevê a média do treino para todos os poços da validação
baseline_pred_0 = np.full(len(target_valid_0), target_train_0.mean())
rmse_baseline_0 = mean_squared_error(target_valid_0, baseline_pred_0) ** 0.5

print(f'REQM do modelo:           {rmse_0:.2f} mil barris')
print(f'REQM do modelo constante: {rmse_baseline_0:.2f} mil barris')
print(f'Redução do erro:          {(1 - rmse_0 / rmse_baseline_0):.1%}')

Volume médio previsto: 92.59 mil barris
Volume médio real:     92.08 mil barris
REQM:                  37.58 mil barris
Verificação de  sanidade:
REQM do modelo:           37.58 mil barris
REQM do modelo constante: 44.29 mil barris
Redução do erro:          15.2%


#### Observações resultados para região 0:
O modelo treinado e validado para região 0, estimou um volume médio de 92,59 mil barris, o que foi um valor próximo ao real de 92,08 mil barris.Isso indica que o modelo não apresentam viés relevante na média. No entanto, ao examinar a raiz do erro padrão (REQM) de 37,58 mil barris o que representa cerca de 41% de volume médio, o que mostra uma baixa precisão. Quando comparamos com o modelo constante á uma redução de 15.2%, indicando que as características fornecem informações uteis, mass com um ganho modesto. Esse resultado é coerentes com a analises de correlação em que encontramos uma relação moderada entre `f2` e product (0,48) e com a correlação fraca de `f0` e `f1`.

### 2.3 Costrução de modelos para as regiões 1 e 2

In [38]:
# função para treinar e avaliar o modelo para cada região
def train_and_evaluate_model(df, region):
    """Passos 2.1 a 2.4 para uma região, com verificação de sanidade (modelo constante). Retorna target_valid, predictions e rmse."""
    features = df.drop(columns=['id', 'product'])
    target = df['product']
    features_train, features_valid, target_train, target_valid = train_test_split(features, target, test_size=0.25, random_state=RANDOM_STATE)

    # Padronização dos dados
    scaler = StandardScaler()
    features_train_scaled = scaler.fit_transform(features_train)
    features_valid_scaled = scaler.transform(features_valid)    

    #modelo de regressão linear
    model = LinearRegression().fit(features_train_scaled, target_train)
    predictions = model.predict(features_valid_scaled)
    rmse = mean_squared_error(target_valid, predictions) ** 0.5

    #verificação de sanidade: modelo constante
    baseline_pred = np.full(len(target_valid), target_train.mean())
    rmse_baseline = mean_squared_error(target_valid, baseline_pred) ** 0.5

    print(f'Região: {region}')
    print("Dimensões do conjunto de treino:", features_train_scaled.shape)
    print("Dimensões do conjunto de validação:", features_valid_scaled.shape)
    print(f'Volume médio previsto:    {predictions.mean():.2f} mil barris')
    print(f'Volume médio real:        {target_valid.mean():.2f} mil barris')
    print(f'REQM do modelo:           {rmse:.2f} mil barris')
    print('Verificação de sanidade')
    print(f'REQM do modelo constante: {rmse_baseline:.2f} mil barris')
    print(f'Redução do erro:          {1 - rmse / rmse_baseline:.1%}\n')

    return target_valid, predictions, rmse, rmse_baseline


In [39]:
# previsões para regiões
target_valid_1, predictions_1, rmse_1, rmse_baseline_1 = train_and_evaluate_model(data['Região 1'], 'Região 1')
target_valid_2, predictions_2, rmse_2, rmse_baseline_2 = train_and_evaluate_model(data['Região 2'], 'Região 2')

Região: Região 1
Dimensões do conjunto de treino: (75000, 3)
Dimensões do conjunto de validação: (25000, 3)
Volume médio previsto:    68.73 mil barris
Volume médio real:        68.72 mil barris
REQM do modelo:           0.89 mil barris
Verificação de sanidade
REQM do modelo constante: 46.02 mil barris
Redução do erro:          98.1%

Região: Região 2
Dimensões do conjunto de treino: (75000, 3)
Dimensões do conjunto de validação: (25000, 3)
Volume médio previsto:    94.97 mil barris
Volume médio real:        94.88 mil barris
REQM do modelo:           40.03 mil barris
Verificação de sanidade
REQM do modelo constante: 44.90 mil barris
Redução do erro:          10.9%



### 2.4 Consolidação dos resultados

In [ ]:
results = {
    'Região 0': {'target': target_valid_0, 'pred': predictions_0, 'rmse': rmse_0},
    'Região 1': {'target': target_valid_1, 'pred': predictions_1, 'rmse': rmse_1},
    'Região 2': {'target': target_valid_2, 'pred': predictions_2, 'rmse': rmse_2},
}

pd.DataFrame({
    region: {'Volume médio previsto': r['pred'].mean(),
             'Volume médio real': r['target'].mean(),
             'REQM': r['rmse']}
    for region, r in results.items()
}).T.round(2)